# Process Trees from COMISET dataset

From Zenodo page:

```
COMISET is a dataset of security events generated by Microsoft Windows systems in two scenarios: LAB a laboratory emulating the infrastructure of a small company, and REAL a computer network commonly used by students at Comillas University.

In the laboratory environment, several attacks were executed involving a variety of techniques and tactics commonly used by the adversaries. 

The monitoring system was implemented to capture the events and label them according to the MITRE ATT&CK matrix.
```

REF: https://www.sciencedirect.com/science/article/pii/S2352340925004512#sec0004

Datasets: https://zenodo.org/records/15375146



In [ ]:
import pandas as pd
import numpy as np
import igraph as ig
from collections import Counter
import matplotlib.pyplot as plt
import tqdm
import json


In [ ]:
# Read file line-by-line into a list of dicts - top 1M lines
with open("Data/top.json", "r") as f:
     _ = [json.loads(line) for line in f]

# Flatten the list of nested objects
_df = pd.json_normalize(_)
_df.columns


In [ ]:
for x in _df.columns:
    if 'process' in x:
        print(x)

In [ ]:
Counter(_df['_source.Processid']).most_common(10)


## Build the process trees

Build edges ```parent --> guid```

Process names are all hashes

We'll use eitherthe ```malicious``` or ```test``` label for our exepriments.


In [ ]:
# Read the entire feather file (prepared from other notebook)
df_tree = pd.read_feather('Data/all_trees.feather')

# Display the first few rows
print(df_tree.head())


In [ ]:
df_roots = df_tree[df_tree.parent=='']
df_tree = df_tree[df_tree.parent!='']
print('number of root nodes:',df_roots.shape[0])

## node dictionaries
child = set(df_tree.guid)
parent = set(df_tree.parent) ## Can be 'None', we'll delete later
nodes = parent.union(child)
print('number of nodes:',len(nodes))
nodes_dict = {v:k for k,v in enumerate(nodes)}
inv_nodes_dict = {k:v for k,v in enumerate(nodes)}


In [ ]:
## build directed graph from edgelist
child = [nodes_dict[x] for x in df_tree.guid]
parent = [nodes_dict[x] for x in df_tree.parent]
edges = np.array([parent,child]).T
G = ig.Graph.TupleList(edges, directed=True)
G = G.simplify()
G.vs['guid'] = [inv_nodes_dict[int(x)] for x in G.vs['name']]
print(G.vcount() ,'nodes and', G.ecount(),'edges')


In [ ]:
## add a few attributes
_dict = dict(zip(df_tree.guid, df_tree.parent))
G.vs["parent"] = [_dict.get(guid, "") for guid in G.vs["guid"]]

_dict = dict(zip(df_tree.guid, df_tree.malicious))
_dict.update(dict(zip(df_roots.guid, df_roots.malicious)))
G.vs["malicious"] = [_dict.get(guid, "") for guid in G.vs["guid"]]

G.vs['color'] = 'black'
for v in G.vs:
    if v['malicious']:
        v['color'] = 'red'

_dict = dict(zip(df_tree.guid, df_tree.process_name))
_dict.update(dict(zip(df_roots.guid, df_roots.process_name)))
G.vs["process"] = [_dict.get(guid, "") for guid in G.vs["guid"]]

_dict = dict(zip(df_tree.guid, df_tree.test))
_dict.update(dict(zip(df_roots.guid, df_roots.test)))
G.vs["test"] = [_dict.get(guid, "") for guid in G.vs["guid"]]

print('malicious label:',Counter(G.vs['malicious']))
print('test label:',Counter(G.vs['test']))

### Process trees and EDA

In [ ]:
## Trees
Trees = G.connected_components(mode="weak")
G.vs['tree'] = Trees.membership
print('largest trees:', Counter(G.vs['tree']).most_common(5))


In [ ]:
## trees with malicious label(s)
## either all or no node are malicious in any given tree
_df = pd.DataFrame( np.array([Trees.graph.vs['tree'] , Trees.graph.vs['malicious']]).T,columns=['tree','malicious'])
print('mean number of malicious nodes:',Counter(_df.groupby(by='tree')['malicious'].mean().to_list()))
tree_is_malicious = np.array(_df.groupby(by='tree')['malicious'].sum().to_list())


In [ ]:
## example - malicious tree
idx = np.where( (np.array(Trees.sizes())==20) & (tree_is_malicious>0) )[0][0]
sg = Trees.subgraph(idx)
ly = sg.layout_reingold_tilford()
ig.plot(sg, layout=ly, vertex_label_size=0)


In [ ]:
# Tree depth distribution
depths = np.empty(len(Trees), dtype="int64")
L = []
for tree_id in tqdm.trange(len(Trees)):
    tree = Trees.subgraph(tree_id)
    root = np.argmin(np.array(tree.degree(mode='in')))
    depth = max(tree.distances(root)[0]) # Trees are padded with extra start root
    depths[tree_id] = depth
    L.append([tree_id, tree.vcount(), sum(tree.vs['malicious']), depth, sum(tree.vs['test']) ])
G.vs['tree_depth'] = [depths[i] for i in G.vs['tree']]
_df = pd.DataFrame(L, columns=['tree','size','malicious','depth', 'test'])
_df['has_malicious'] = (_df['malicious']>0)
_df.head(5)


#### It turns out that tree depth is a strong feature to identify malicious trees ...


In [ ]:
ax = _df.boxplot(column="depth", by="has_malicious", showfliers=True)
ax.set_yscale("log")


In [ ]:
## most tree are shallow, a few are very deep
print('common depth:\n',Counter(depths).most_common(5))
print('deepest:',max(depths))
## max depth - non-malicious trees
print('deepest non-malicious:',max(_df[_df.has_malicious==False]['depth']))


## Matching algorithm

In [ ]:
## our matching algorithm
import igraph_io as igio
import fast_match as fm  ## fast version


In [ ]:
## select trees of depth 3+
malicious_trees = list(_df[ (_df.depth>=3) & (_df.has_malicious) ]['tree'])
non_malicious_trees = list(_df[ (_df.depth>=3) & (_df.has_malicious == False) ]['tree'])
Trees.graph.vs['label_color'] = Trees.graph.vs['color']


In [ ]:
%%time
Graphs = []
TreeData = []
nmt = 0

for i in malicious_trees:
    sg = Trees.subgraph(i)
    if sg.vcount() == (sg.ecount()+1):
        Graphs.append(sg)
        TreeData.append(igio.igraph_to_treedata(sg, phi_name='process'))
        nmt += 1

for i in non_malicious_trees:
    sg = Trees.subgraph(i)
    Graphs.append(sg)
    TreeData.append(igio.igraph_to_treedata(sg, phi_name='process'))

## encode all trees
fast = fm.FastTreePathMatcher()
fast.fit_encoder(TreeData)
enc = [fast.encode_tree(t) for t in TreeData]
       

In [ ]:
%%time
## compare all non-malicious tree with every malicious one (the templates)
Sim = np.zeros(shape=(nmt, len(non_malicious_trees)))
Paths = []
for i in range(nmt):
    for j in range(len(non_malicious_trees)):
        _, score = fast.predict_encoded(enc[i], enc[j+nmt])
        Sim[i,j] = score
Sim.shape
        

In [ ]:
## pick top similarity pair and visualize common path
(bad, nonbad) = np.unravel_index(np.argmax(Sim), Sim.shape)
sg1 = Graphs[bad]
sg2 = Graphs[nonbad+nmt]
_sg2 = igio.igraph_to_treedata(sg2, phi_name='process')
_sg1 = igio.igraph_to_treedata(sg1, phi_name='process')
fast = fm.FastTreePathMatcher()
fast.fit(_sg1,_sg2)
paths, score = fast.predict()

print('length of matching path:',score)


In [ ]:
## plot malicious tree; highlight path
path = [int(x[0]) for x in paths]
sg1.vs['vertex_size'] = 8
sg1.vs['vertex_label_size'] = 1
for i in path:
    sg1.vs[i]['vertex_size'] = 1
    sg1.vs[i]['vertex_label_size'] = 20
print('number of nodes:',sg1.vcount())
ly = sg1.layout_reingold_tilford()
ig.plot(sg1, 
        bbox=(800,600), margin=100, layout=ly,
        vertex_size=sg1.vs['vertex_size'], 
        vertex_label=sg1.vs['process'], 
        vertex_label_size=sg1.vs['vertex_label_size'], 
        edge_color='lightgrey', edge_arrow_size=0)


In [ ]:
## plot matching non-malicious tree
path = [int(x[1]) for x in paths]
sg2.vs['vertex_size'] = 8
sg2.vs['vertex_label_size'] = 0
for i in path:
    sg2.vs[i]['vertex_size'] = 1
    sg2.vs[i]['vertex_label_size'] = 15
print('number of nodes:',sg2.vcount())
ly = sg2.layout_reingold_tilford()
ig.plot(sg2,
        bbox=(700,500), layout=ly, margin=100, 
        vertex_size=sg2.vs['vertex_size'], 
        vertex_label=sg2.vs['process'], 
        vertex_label_size=sg2.vs['vertex_label_size'], 
        edge_color='lightgrey', edge_arrow_size=0)
